In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc pandas qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Quantum Portfolio Optimizer: A Qiskit Function by Global Data Quantum



*Lihat [referensi API](https://docs.quantum.ibm.com/api/functions/global-data-quantum-optimizer)*

> **Note:** Qiskit Functions adalah fitur eksperimental yang hanya tersedia untuk pengguna IBM Quantum&reg; Premium Plan, Flex Plan, dan On-Prem (via IBM Quantum Platform API) Plan. Fitur ini dalam status rilis pratinjau dan dapat berubah sewaktu-waktu.
## Gambaran Umum
Quantum Portfolio Optimizer adalah sebuah Qiskit Function yang menangani masalah optimasi portofolio dinamis, sebuah masalah standar dalam keuangan yang bertujuan untuk menyeimbangkan kembali investasi periodik di sejumlah aset, untuk memaksimalkan keuntungan dan meminimalkan risiko. Dengan menggunakan teknik optimasi kuantum mutakhir, fungsi ini menyederhanakan prosesnya sehingga pengguna, tanpa keahlian di bidang komputasi kuantum, bisa memanfaatkan keunggulannya dalam menemukan trajektori investasi yang optimal. Ideal untuk manajer portofolio, peneliti keuangan kuantitatif, dan investor individu, alat ini memungkinkan back-testing strategi trading dalam optimasi portofolio.
## Deskripsi Fungsi
Fungsi Quantum Portfolio Optimizer menggunakan algoritma Variational Quantum Eigensolver (VQE) untuk menyelesaikan masalah Quadratic Unconstrained Binary Optimization (QUBO), yang menangani masalah optimasi portofolio dinamis. Pengguna cukup menyediakan data harga aset dan mendefinisikan batasan investasi, lalu fungsi ini menjalankan proses optimasi kuantum yang mengembalikan sekumpulan trajektori investasi yang optimal.

Proses ini terdiri dari empat tahap utama. Pertama, data input dipetakan ke masalah yang kompatibel dengan kuantum, membangun QUBO dari masalah optimasi portofolio dinamis, dan mengubahnya menjadi operator kuantum (Ising Hamiltonian). Selanjutnya, masalah input dan algoritma VQE diadaptasi agar dapat dijalankan di perangkat keras kuantum. Algoritma VQE kemudian dijalankan di perangkat keras kuantum, dan akhirnya, hasilnya diproses pasca-komputasi untuk memberikan trajektori investasi yang optimal. Sistem ini juga mencakup pasca-pemrosesan yang peka terhadap noise (berbasis [SQD](/guides/qiskit-addons-sqd)) untuk memaksimalkan kualitas output.

Qiskit Function ini didasarkan pada [naskah yang sudah dipublikasikan](https://arxiv.org/abs/2412.19150) oleh Global Data Quantum.
![Visualisasi alur kerja fungsi](../docs/images/guides/global-data-quantum-optimizer/function_workflow.svg)
# Mulai
Autentikasi menggunakan [kunci API](https://quantum.cloud.ibm.com/) kamu dan pilih Qiskit Function sebagai berikut. (Cuplikan ini mengasumsikan kamu sudah [menyimpan akunmu](/guides/functions#install-qiskit-functions-catalog-client) ke lingkungan lokal.)

In [ ]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# Access function
dpo_solver = catalog.load("global-data-quantum/quantum-portfolio-optimizer")

## Contoh: Optimasi portofolio dinamis dengan tujuh aset
Contoh ini menunjukkan cara mengeksekusi fungsi dynamic portfolio optimization (DPO) dan menyesuaikan pengaturannya untuk performa optimal. Ini mencakup langkah-langkah detail untuk menyetel parameter agar mencapai hasil yang diinginkan.

Kasus ini melibatkan tujuh aset, empat langkah waktu, dan empat Qubit resolusi, sehingga menghasilkan total kebutuhan 112 Qubit.
### 1. Baca aset yang termasuk dalam portofolio.
Jika semua aset dalam portofolio disimpan di folder pada path tertentu, kamu bisa memuatnya ke dalam `pandas.DataFrame` dan mengonversinya ke objek format `dict` menggunakan fungsi berikut.

In [ ]:
import os
import glob
import pandas as pd


def read_and_join_csv(file_pattern):
    """
    Reads multiple CSV files matching the file pattern and combines them into a single DataFrame.

    Parameters:
    file_pattern (str): The pattern to match CSV files.

    Returns:
    pd.DataFrame: Combined DataFrame with data from all CSV files.
    """
    # Find all files matching the pattern
    csv_files = glob.glob(file_pattern)
    # Get the base file names without the .csv extension
    file_names = [os.path.basename(f).replace(".csv", "") for f in csv_files]
    # Read each CSV file into a DataFrame and set the first column as the index
    df_list = [pd.read_csv(f).set_index("Unnamed: 0") for f in csv_files]

    # Rename columns in each DataFrame to the base file names
    for df, name in zip(df_list, file_names):
        df.columns = [name]

    # Combine all DataFrames into one by merging them side by side
    combined_df = pd.concat(df_list, axis=1)
    return combined_df


file_pattern = "route/to/folder/with/assets/data/*.csv"
assets = read_and_join_csv(file_pattern).to_dict()

Untuk contoh ini, kami menggunakan aset [8801.T](https://finance.yahoo.com/quote/8801.T), [CLF](https://finance.yahoo.com/quote/CLF), [GBPJPY](https://finance.yahoo.com/quote/GBPJPY), [ITX.MC](https://finance.yahoo.com/quote/ITX.MC), [META](https://finance.yahoo.com/quote/META), [TMBMKDE-10Y](https://finance.yahoo.com/quote/TMBMKDE-10Y), dan [XS2239553048](https://finance.yahoo.com/quote/XS2239553048). Gambar berikut mengilustrasikan data yang digunakan dalam contoh ini, menunjukkan evolusi harga penutupan harian aset dari 1 Januari hingga 1 September 2023.

Dalam contoh ini, untuk memastikan keseragaman antar tanggal, kami mengisi hari non-trading dengan harga penutupan dari tanggal sebelumnya yang tersedia. Kami menerapkan langkah ini karena aset yang dipilih berasal dari berbagai pasar dengan hari trading yang berbeda-beda, sehingga penting untuk menstandarisasi dataset demi konsistensi.
![Visualisasi data historis aset](../docs/images/guides/global-data-quantum-optimizer/assets.avif)
### 2. Definisikan masalah.
Tentukan spesifikasi masalah dengan mengonfigurasi parameter dalam dictionary `qubo_settings`.

In [ ]:
qubo_settings = {
    "nt": 4,
    "nq": 4,
    "dt": 30,
    "max_investment": 25,
    "risk_aversion": 1000.0,
    "transaction_fee": 0.01,
    "restriction_coeff": 1.0,
}

### 3. Definisikan pengaturan optimizer dan ansatz (Opsional)
Secara opsional, tentukan persyaratan khusus untuk proses optimasi, termasuk pemilihan optimizer dan parameternya, serta spesifikasi primitive dan konfigurasinya.

Untuk Tailored Ansatz, ukuran populasi yang dipilih didasarkan pada eksperimen sebelumnya yang menunjukkan bahwa nilai ini menghasilkan optimasi yang stabil dan efisien.

Dalam kasus Real Amplitudes Ansatz, kamu bisa mengikuti hubungan linear antara ``population_size`` dan jumlah Qubit dalam Circuit. Sebagai aturan perkiraan, disarankan untuk menggunakan **minimum** ``population_size ~ 0.8 * n_qubits`` untuk ansatz `real_amplitudes`.

Diharapkan bahwa Optimized Real Amplitudes akan memiliki performa optimasi yang lebih baik daripada ansatz Real Amplitudes. Namun, jumlah variabel yang perlu dioptimalkan dalam ansatz ini meningkat jauh lebih cepat daripada pada kasus Real Amplitudes (lihat [naskah](https://arxiv.org/pdf/2412.19150)). Jadi, untuk masalah besar, Optimized Real Amplitudes memerlukan lebih banyak eksekusi Circuit. Optimized Real Amplitudes kemungkinan berguna untuk masalah yang memerlukan hingga 100 Qubit, tetapi disarankan untuk berhati-hati saat mengatur parameter ``population_size``. Sebagai contoh scale-up dalam ``population_size``, tabel sebelumnya menunjukkan bahwa untuk masalah 84 Qubit, Optimized Real Amplitudes memerlukan 120 ``population_size``, sementara untuk masalah 56 Qubit, ``population_size`` sebesar 40 sudah cukup.

In [ ]:
optimizer_settings = {
    "de_optimizer_settings": {
        "num_generations": 20,
        "population_size": 90,
        "recombination": 0.4,
        "max_parallel_jobs": 5,
        "max_batchsize": 4,
        "mutation_range": [0.0, 0.25],
    },
    "optimizer": "differential_evolution",
    "primitive_settings": {
        "estimator_shots": 25_000,
        "estimator_precision": None,
        "sampler_shots": 100_000,
    },
}

Juga mungkin untuk memilih ansatz tertentu. Berikut menggunakan ansatz ``'Tailored'``.

In [ ]:
ansatz_settings = {
    "ansatz": "tailored",
    "multiple_passmanager": False,
}

### 4. Jalankan masalah.

In [ ]:
dpo_job = dpo_solver.run(
    assets=assets,
    qubo_settings=qubo_settings,
    optimizer_settings=optimizer_settings,
    ansatz_settings=ansatz_settings,
    backend_name="<backend name>",
    previous_session_id=[],
    apply_postprocess=True,
)

### 5. Ambil hasil
Fungsi ini mengembalikan dictionary dengan trajektori investasi yang diurutkan dari terendah ke tertinggi berdasarkan nilai fungsi objektifnya (lihat bagian [Output](https://docs.quantum.ibm.com/api/functions/global-data-quantum-optimizer#output) dari referensi API). Kumpulan hasil ini memungkinkan identifikasi trajektori dengan biaya terendah dan evaluasi investasi yang sesuai. Selain itu, ini menyediakan analisis berbagai trajektori, memfasilitasi pemilihan yang paling sesuai dengan kebutuhan atau tujuan tertentu. Fleksibilitas ini memastikan bahwa pilihan bisa disesuaikan dengan berbagai preferensi atau skenario.
Mulailah dengan menyajikan strategi hasil yang mencapai biaya objektif terendah yang ditemukan selama proses.

In [ ]:
# Get the results of the job
dpo_result = dpo_job.result()

# Show the solution strategy
dpo_result["result"]

{'time_step_0': {'8801.T': 0.11764705882352941,
  'ITX.MC': 0.20588235294117646,
  'META': 0.38235294117647056,
  'GBPJPY=X': 0.058823529411764705,
  'TMBMKDE-10Y': 0.0,
  'CLF': 0.058823529411764705,
  'XS2239553048': 0.17647058823529413},
 'time_step_1': {'8801.T': 0.11428571428571428,
  'ITX.MC': 0.14285714285714285,
  'META': 0.2,
  'GBPJPY=X': 0.02857142857142857,
  'TMBMKDE-10Y': 0.42857142857142855,
  'CLF': 0.0,
  'XS2239553048': 0.08571428571428572},
 'time_step_2': {'8801.T': 0.0,
  'ITX.MC': 0.09375,
  'META': 0.3125,
  'GBPJPY=X': 0.34375,
  'TMBMKDE-10Y': 0.0,
  'CLF': 0.0,
  'XS2239553048': 0.25},
 'time_step_3': {'8801.T': 0.3939393939393939,
  'ITX.MC': 0.09090909090909091,
  'META': 0.12121212121212122,
  'GBPJPY=X': 0.18181818181818182,
  'TMBMKDE-10Y': 0.0,
  'CLF': 0.0,
  'XS2239553048': 0.21212121212121213}}

Afterwards, using the metadata, you can access the results of all the sampled strategies. You can thereby further analyze the alternative trajectories returned by the optimizer. To do this, read the dictionary stored in `dpo_result['metadata']['all_samples_metrics']`, which contains not only additional information about the optimal strategy, but also details of the other candidate strategies evaluated during the optimization.

The following example shows how to read this information using `pandas` to extract key metrics associated with the optimal strategy. These include Restriction Deviation, Sharpe Ratio, and the corresponding investment return.

In [ ]:
# Convert metadata to a DataFrame
df = pd.DataFrame(dpo_result["metadata"]["all_samples_metrics"])

# Find the minimum objective cost
min_cost = df["objective_costs"].min()
print(f"Minimum Objective Cost Found: {min_cost:.2f}")

# Extract the row with the lowest cost
best_row = df[df["objective_costs"] == min_cost].iloc[0]

# Display the results associated with the best solution
print("Best Solution:")
print(f"  - Restriction Deviation: {best_row['rest_breaches']}%")
print(f"  - Sharpe Ratio: {best_row['sharpe_ratios']:.2f}")
print(f"  - Return: {best_row['returns']}")

Minimum Objective Cost Found: -3.78
Best Solution:
  - Restriction Deviation: 40.0
  - Sharpe Ratio: 24.82
  - Return: 0.46


Setelah itu, menggunakan metadata, kamu bisa mengakses hasil dari semua strategi yang disampel. Kamu bisa menganalisis lebih lanjut trajektori alternatif yang dikembalikan oleh optimizer. Untuk melakukan ini, baca dictionary yang tersimpan di `dpo_result['metadata']['all_samples_metrics']`, yang berisi tidak hanya informasi tambahan tentang strategi optimal, tetapi juga detail tentang strategi kandidat lainnya yang dievaluasi selama optimasi.

Contoh berikut menunjukkan cara membaca informasi ini menggunakan `pandas` untuk mengekstrak metrik kunci yang terkait dengan strategi optimal. Ini termasuk Restriction Deviation, Sharpe Ratio, dan return investasi yang sesuai.